# SSSP and Modified Dijkstra's Algorithm

This lecture focuses on solving the **Single Source Shortest Path (SSSP)** problem in graphs that contain **negative edge weights** but lack **negative cycles**. It explains why the standard Dijkstra's algorithm fails in these scenarios and introduces a modified version to ensure correctness.

---

## 1. Context and Problem Overview
Throughout the study of shortest paths, different algorithms are chosen based on the nature of the edge weights:
* **Uniform Edge Weights:** **Breadth-First Search (BFS)** is the most efficient choice.
* **Non-negative Edge Weights:** Standard **Dijkstra’s Algorithm** is the standard approach.
* **Negative Edge Weights (No Negative Cycles):** Requires a **Modified Dijkstra** or other advanced algorithms.

### Negative Cycles
A **negative cycle** is a cycle where the sum of the participating edge weights is less than zero.
* **Problem:** If a graph has a negative cycle, you could theoretically traverse it infinitely to keep reducing the path weight, meaning a "shortest path" might not be well-defined.
* **Assumption:** This lecture assumes the graph **does not** contain negative cycles, even though it may contain negative individual edge weights.

---

## 2. Failure of Standard Dijkstra's Algorithm
Standard Dijkstra's algorithm relies on the **Greedy Property**: once a vertex is extracted from the priority queue (marked as "done"), its shortest path distance is finalized and never updated again.

### Why It Fails with Negative Weights
In the presence of negative weights, a path that initially looks expensive (and is processed later) might eventually become the shortest path by using a high-magnitude negative edge. Because the standard algorithm never "re-visits" a vertex once it is processed, it misses these improvements.


**Example Scenario:**
1.  Path $S \to A \to C$ has weight 3.
2.  Path $S \to B$ has weight 10.
3.  Standard Dijkstra finishes processing $C$ with a distance of 3.
4.  Later, it finds an edge $B \to C$ with weight -10.
5.  The new distance to $C$ via $B$ would be $10 + (-10) = 0$.
6.  **Standard Dijkstra fails** because $C$ has already been removed from the queue and cannot be updated.

---

## 3. Modified Dijkstra’s Algorithm
The core fix is to allow vertices to **return to the priority queue** if a shorter path to them is discovered later.

### Key Changes
* **Initialization:** Instead of putting all vertices into the queue at once, only the source vertex $S$ with distance 0 is inserted.
* **Re-insertion:** During edge relaxation, if a shorter path to a vertex $V$ is found, $V$ is added back to the priority queue—even if it was previously extracted.

### Pseudocode Logic
```
Procedure Modified_Dijkstra(Graph, Source):
    Initialize all distances to Infinity
    Set distance[Source] = 0
    
    Create an empty Priority Queue (PQ)
    Insert (Source, 0) into PQ
    
    While PQ is not empty:
        Extract the pair (U, current_dist) with the minimum distance
        
        // Lazy Deletion Check
        If current_dist > distance[U]:
            Continue to next iteration (skip obsolete value)
            
        For each neighbor V of U with edge weight W:
            If distance[U] + W < distance[V]:
                Set distance[V] = distance[U] + W
                Insert (V, distance[V]) into PQ
```

---

## 4. Implementation Details: Lazy Deletion
Standard priority queues (like those in Python's `heapq` or some C++ libraries) do not always support an efficient `decrease_key` operation. To handle this, the lecture suggests **Lazy Deletion**:

1.  **Multiple Entries:** When a shorter path to a vertex is found, don't try to find and update its existing entry in the queue. Simply **insert a new entry** with the updated (smaller) distance.
2.  **The Stale Check:** When you extract a vertex from the queue, compare its distance in the queue to its current value in the distance array.
3.  **Action:** If the distance from the queue is larger than the recorded distance, it is a **"stale" or obsolete value** and should be ignored.

---

## 5. Performance and Limitations
### Running Time
* **Non-negative weights:** The performance is similar to the standard version.
* **Negative weights:** The running time can be significantly worse in the worst-case (exponential in some adversarial examples) because vertices can be processed multiple times. However, it provides the **correct answer** where the standard version fails.

### Negative Cycles
If the graph contains a **negative cycle**, the modified Dijkstra's algorithm can fall into an **infinite loop**. The distance values will keep decreasing every time the cycle is traversed, and vertices will keep being re-inserted into the queue indefinitely.

---

## Summary of Takeaways
* **Standard Dijkstra** is incorrect for graphs with negative edge weights because it assumes a vertex's distance is final once extracted from the queue.
* **Modified Dijkstra** fixes this by allowing vertices to be re-inserted into the priority queue whenever their distance is updated.
* **Lazy Deletion** is a practical implementation technique using a standard priority queue where stale entries are ignored upon extraction.
* **Warning:** Neither version of Dijkstra is suitable for graphs with **negative cycles**; such graphs require specific detection and handling (like the Bellman-Ford algorithm, discussed in later modules).